<a href="https://colab.research.google.com/github/Talzablev/B6/blob/main/HW3_Turtles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. SETUP, IMPORTS & GLOBAL CONFIGURATION , CLOUD SERVICES & API WRAPPER




In [1]:
print("Installing required libraries...")
!pip install -q google-generativeai pandas matplotlib plotly requests PyPDF2 nltk ipywidgets pillow transformers
# --- Standard Libraries ---
import re
import datetime
from datetime import timedelta
import io
import time
import random
import logging


# --- Data & Visualization ---
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import requests
from PIL import Image

# --- UI Widgets ---
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown, HTML

# --- NLP & AI ---
from nltk.stem import PorterStemmer
import google.generativeai as genai

# --- Cloud & Database ---
from google.colab import userdata

from transformers import pipeline
from google.colab import drive
drive.mount('/content/drive')

print("\nLoading Global Configurations...")

# A. Load Secrets securely from Colab's Secrets Manager (🔑)
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print("✅ Gemini API Key loaded successfully.")
except userdata.SecretNotFoundError:
    print("❌ Error: Missing GEMINI_API_KEY. Please add it to the Colab Secrets tab.")
    GEMINI_API_KEY = None

# B. Configure Gemini AI
if GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)
    print("✅ Gemini AI configured successfully.")

# C. Load Local Vision Model
print("📥 Loading Local Vision Model (this may take a minute)...")
disease_classifier = pipeline("image-classification", model="/content/drive/MyDrive/final-basil-model")
print("✅ Local Vision Model loaded successfully!")

# D. IoT Server Configuration (Render)
BASE_URL = "https://server-cloud-v645.onrender.com/"
print("✅ IoT Server URL loaded.")


Installing required libraries...


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Loading Global Configurations...
✅ Gemini API Key loaded successfully.
✅ Gemini AI configured successfully.
📥 Loading Local Vision Model (this may take a minute)...


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

✅ Local Vision Model loaded successfully!
✅ IoT Server URL loaded.


In [2]:
# Custom Firebase Wrapper
class FirebaseApplication:
    """A lightweight wrapper to communicate with Firebase REST API using requests."""
    def __init__(self, database_url, authentication=None):
        self.db_url = database_url.rstrip('/')

    def child(self, path_segment):
        return FirebaseApplication(f"{self.db_url}/{path_segment}")

    def get(self, path, name=None):
        url = f"{self.db_url}{path}/{name}.json" if name else f"{self.db_url}{path}.json"
        response = requests.get(url)
        return response.json() if response.status_code == 200 else None

    def put(self, path, name, data):
        url = f"{self.db_url}{path}/{name}.json"
        response = requests.put(url, json=data)
        return response.json() if response.status_code == 200 else None

    def post(self, path, data):
        url = f"{self.db_url}{path}.json"
        response = requests.post(url, json=data)
        return response.json() if response.status_code == 200 else None

    def delete(self, path, name=None):
        url = f"{self.db_url}{path}/{name}.json" if name else f"{self.db_url}{path}.json"
        response = requests.delete(url)
        return response.json() if response.status_code == 200 else None

# Initialize Firebase Connection
FIREBASE_URL = 'https://basil-plant-disease-default-rtdb.firebaseio.com/'
try:
    FBconn = FirebaseApplication(FIREBASE_URL, None)
    print("✅ Firebase REST connection established successfully.")
except Exception as e:
    print(f"❌ Firebase connection error: {e}")

✅ Firebase REST connection established successfully.


### Constants

In [3]:

# Unified Design System
# A single color palette applied to ALL screens for a consistent look & feel.
COLOR_PRIMARY   = "#2ecc71"   # Green  – main brand / success actions
COLOR_SECONDARY = "#27ae60"   # Dark green – headings, borders
COLOR_ACCENT    = "#3498db"   # Blue   – info / IoT data
COLOR_WARNING   = "#f39c12"   # Amber  – temperature / alerts
COLOR_DANGER    = "#e74c3c"   # Red    – errors / high-temp
COLOR_SURFACE   = "#f8f9fa"   # Light grey – card backgrounds
COLOR_TEXT      = "#2c3e50"   # Dark   – all body text


### Logger Setup

In [4]:
# Configures a hidden file logger for developer debugging. No prints to the UI.
logger = logging.getLogger('TurtleGrow_UI')
logger.setLevel(logging.DEBUG)

# Stop Colab from printing logs to the screen
logger.propagate = False

# Prevent duplicate handlers if the cell is run multiple times
if not logger.handlers:
    file_handler = logging.FileHandler('system_debug.log')
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

# Silence noisy third-party loggers (e.g., Tornado HTTP 503 errors, Google API)
# to ensure the UI remains perfectly clean for the end user.
logging.getLogger('tornado.access').setLevel(logging.CRITICAL)
logging.getLogger('tornado.application').setLevel(logging.CRITICAL)
logging.getLogger('tornado.general').setLevel(logging.CRITICAL)

logging.getLogger('google').setLevel(logging.ERROR)
logging.getLogger('google_auth_httplib2').setLevel(logging.ERROR)
logging.getLogger('urllib3').setLevel(logging.ERROR)

## 2. RAG System Class Definition

> Add blockquote



In [5]:
class RAGSystem:
    def __init__(self):
        print("Initializing RAG System...")

        # Map document IDs to their original source URLs
        self.doc_id_mapping = {
            1: "https://doi.org/10.3390/plants9050654",
            2: "https://doi.org/10.1094/pdis.1997.81.2.124",
            3: "https://doi.org/10.21273/hortsci09778-16",
            4: "https://doi.org/10.3732/apps.1300032",
            5: "https://doi.org/10.21273/horttech03849-17"
        }

        # Initialize Porter Stemmer and define Stop Words
        self.stemmer = PorterStemmer()
        self.stop_words = {
            "the", "and", "of", "in", "to", "a", "is", "for", "on", "with", "by",
            "from", "at", "as", "it", "he", "she", "they", "we", "you", "that", "this",
            "but", "or", "not", "has", "have", "had", "do", "does", "did", "can",
            "will", "would", "should", "could", "may", "might", "must"
        }

        # Updated Corpus - Real abstracts from the 5 academic articles
        self.articles_corpus = {
            1: "Black spot is a major foliar disease of sweet basil (Ocimum basilicum) present in a typical cultivation area of northern Italy. In this study, 15 Colletotrichum isolates obtained from sweet basil plants with symptoms of black spot were characterized. Analysis revealed C. ocimi of the C. destructivum species complex.",
            2: "Sweet basil (Ocimum basilicum L.) is an economically important herb crop in several Mediterranean countries. The intensified use of cultivation systems, coupled with increasing restrictions on the use of fungicides, led to severe epidemics. The recent widespread outbreaks of Fusarium wilt and black spot on basil clearly show that effective control measures are needed.",
            3: "Sweet basil (Ocimum basilicum) is the most economically important culinary herb in the United States. In 2007, a new disease, basil downy mildew (BDM), caused by the oomycete pathogen Peronospora belbahrii, was introduced and has since caused significant losses. Basil species with higher stomatal densities had higher downy mildew incidence.",
            4: "Demand for fresh-market sweet basil continues to increase, but in 2009 a new pathogen emerged, threatening commercial production and leading to high crop losses. This study describes a simple and effective staining method for rapid microscopic detection of basil downy mildew (Peronospora belbahrii) from leaves of basil.",
            5: "Different basils (Ocimum sp.) and cultivars were evaluated for susceptibility to basil downy mildew (Peronospora belbahrii). At the end of each growing season, seed was collected from individual plants. Real-time PCR analysis of seed collected from various infected plants detected P. belbahrii on seed of multiple basil lines."
        }

        # Check Firebase for existing index; create dynamically if missing
        print("Checking Firebase Realtime Database for existing inverted index...")

        try:
            cloud_index = FBconn.get('/search_engine', 'inverted_index')
        except Exception as e:
            cloud_index = None
            print(f"Warning: Could not connect to Firebase for index check: {e}")

        self.inverted_index = {}

        if cloud_index is not None:
            print("Index found in Firebase Cloud Cache! Loaded directly.")
            if isinstance(cloud_index, dict):
                cloud_index = [v for k, v in cloud_index.items()]
            for item in cloud_index:
                self.inverted_index[item["term"]] = item["DocIDs"]
        else:
            print("Index not found in Firebase. Creating dynamically, applying Stemmer, and uploading...")
            raw_index_data = {}

            for doc_id, text in self.articles_corpus.items():
                words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
                for word in words:
                    if word not in self.stop_words:
                        stemmed_word = self.stemmer.stem(word)
                        if stemmed_word not in raw_index_data:
                            raw_index_data[stemmed_word] = set()
                        raw_index_data[stemmed_word].add(doc_id)

            index_for_cloud = [{"term": term, "DocIDs": sorted(list(docs))} for term, docs in raw_index_data.items()]

            try:
                FBconn.put('/search_engine', 'inverted_index', index_for_cloud)
                print("Optimized Index successfully saved to Firebase.")
            except Exception as e:
                print(f"Warning: Could not save index to Firebase: {e}")

            for item in index_for_cloud:
                self.inverted_index[item["term"]] = item["DocIDs"]

        # Dynamically detect the first available generative model to avoid 404 errors
        print("Detecting available Gemini models...")
        self.gemini_model = None
        try:
            for m in genai.list_models():
                if 'generateContent' in m.supported_generation_methods:
                    self.gemini_model = genai.GenerativeModel(m.name.replace('models/', ''))
                    print(f"✅ Successfully initialized model: {m.name}")
                    break
        except Exception as e:
            print(f"Error detecting Gemini model: {e}")

    def retrieve_documents(self, query_keywords, top_n=2):
        # We will use a dictionary to keep track of scores: {doc_id: score}
        doc_scores = {}

        for keyword in query_keywords:
            stemmed_keyword = self.stemmer.stem(keyword.lower())

            # If the word is in the index, give 1 point to each document that contains it
            if stemmed_keyword in self.inverted_index:
                for doc_id in self.inverted_index[stemmed_keyword]:
                    doc_scores[doc_id] = doc_scores.get(doc_id, 0) + 1

        # Sort documents by their SCORE
        # item[1] is the score. reverse=True means highest score first.
        sorted_docs_by_score = sorted(doc_scores.items(), key=lambda item: item[1], reverse=True)

        top_documents = []

        # Take the top N documents with the highest scores
        for doc_id, score in sorted_docs_by_score[:top_n]:
            if doc_id in self.articles_corpus:
                # You can also add the score to the output if you want to see it
                top_documents.append({
                    'id': doc_id,
                    'text': self.articles_corpus[doc_id],
                    'relevance_score': score
                })

        return top_documents

    def generate_rag_response(self, question, context):
        if not self.gemini_model:
            return "AI Model is not initialized. Please check API key permissions."

        if not context:
            return "I could not find an answer in the provided articles. Please ask something else."

        context_text = "\n\n".join([f"Source [{doc['id']}] ({self.doc_id_mapping.get(doc['id'], 'Unknown URL')}): {doc['text']}" for doc in context])

        prompt = f"""You are an agricultural expert specializing in plant diseases.
        Given the following context from academic articles, answer the user's question comprehensively and accurately.
        If the answer is not in the context, do not guess, just state that you don't know based on the provided texts.

        Context:
        {context_text}

        Question: {question}
        Answer:"""

        try:
            response = self.gemini_model.generate_content(prompt)
            generated_answer = response.text.strip()
            if not generated_answer:
                return "I could not generate an answer according to the articles."
            return generated_answer
        except Exception as e:
            print(f"Error generating content: {e}")
            return "I encountered an error connecting to the AI model. Please try again."

    def get_existing_queries(self, fb_connection):
        try:
            all_rag_results = fb_connection.get('/rag_search_results', None)
            if all_rag_results:
                return {value.get('query').lower(): key for key, value in all_rag_results.items() if value.get('query')}
            return {}
        except Exception as e:
            print(f"Error fetching existing queries from Firebase: {e}")
            return {}

## 3. Instantiate and Launch RAG System

In [6]:
rag_instance = RAGSystem()
print("\n✅ RAG System is fully initialized and ready to use")

Initializing RAG System...
Checking Firebase Realtime Database for existing inverted index...
Index found in Firebase Cloud Cache! Loaded directly.
Detecting available Gemini models...
✅ Successfully initialized model: models/gemini-2.5-flash

✅ RAG System is fully initialized and ready to use


In [7]:

print("--- Inverted Index Content ---")
for term, doc_ids in rag_instance.inverted_index.items():
    print(f"Term: '{term}' -> Found in Doc IDs: {doc_ids}")

--- Inverted Index Content ---
Term: 'basil' -> Found in Doc IDs: [1, 2, 3, 5]
Term: 'diseas' -> Found in Doc IDs: [1, 2, 3, 4, 5]
Term: 'downi' -> Found in Doc IDs: [1, 5]
Term: 'mildew' -> Found in Doc IDs: [1, 5]
Term: 'manag' -> Found in Doc IDs: [1, 2, 3, 5]
Term: 'infect' -> Found in Doc IDs: [1, 3, 5]
Term: 'resist' -> Found in Doc IDs: [1, 2, 3, 5]
Term: 'peronospora' -> Found in Doc IDs: [5]
Term: 'belbahrii' -> Found in Doc IDs: [5]
Term: 'sporul' -> Found in Doc IDs: [3, 5]
Term: 'chlorosi' -> Found in Doc IDs: [5]
Term: 'oomycet' -> Found in Doc IDs: [5]
Term: 'spore' -> Found in Doc IDs: [5]
Term: 'greenhous' -> Found in Doc IDs: [2, 3, 5]
Term: 'inoculum' -> Found in Doc IDs: [3, 5]
Term: 'epidemiolog' -> Found in Doc IDs: [5]
Term: 'fusarium' -> Found in Doc IDs: [2]
Term: 'wilt' -> Found in Doc IDs: [2]
Term: 'mould' -> Found in Doc IDs: [3]
Term: 'white' -> Found in Doc IDs: [3]
Term: 'epidermi' -> Found in Doc IDs: [5]
Term: 'temperatur' -> Found in Doc IDs: [3, 5]


## 4. Testing & Populating the RAG System

In [8]:
print("--- Testing the RAG Pipeline & Database ---")

# 1. Define a test query
test_query = "What causes downy mildew in basil and what are the symptoms?"
print(f"\nUser Query: {test_query}")

# 2. Dynamically extract keywords and search the Inverted Index
query_keywords = re.findall(r'\b[a-zA-Z]+\b', test_query)
retrieved_context = rag_instance.retrieve_documents(query_keywords, top_n=2)
print(f"Found {len(retrieved_context)} relevant documents using the Search Engine.")

# 3. Generate the smart answer using Gemini
answer = rag_instance.generate_rag_response(test_query, retrieved_context)
print(f"\nGenerated Answer:\n{answer}")

# 4. Save the new interaction dynamically to Firebase (Using POST for a unique ID)
try:
    context_text = "\n".join([f"Doc {doc['id']}: {doc['text']}" for doc in retrieved_context])
    rag_data_new = {
        'query': test_query,
        'answer': answer,
        'context': context_text,
        'timestamp': datetime.datetime.now().isoformat()
    }

    # Using the post method from your wrapper
    FBconn.post('/rag_search_results', rag_data_new)
    print("\n✅ Successfully saved the query and answer to Firebase.")
except Exception as e:
    print(f"\n❌ Error saving to Firebase: {e}")

print("\n--- Fetching Search History from Firebase ---")
try:
    # Fetch all search history from the cloud
    all_results = FBconn.get('/rag_search_results', None)

    if all_results:
        print(f"Found {len(all_results)} saved searches in the database!\n")
        # Iterate through the results and print the saved queries and answers
        for key, data in all_results.items():
            print(f"Cloud ID: {key}")
            print(f"Query: {data.get('query', 'N/A')}")
            print("-" * 40)
    else:
        print("The database is currently empty.")
except Exception as e:
    print(f"Error reading from Firebase: {e}")

--- Testing the RAG Pipeline & Database ---

User Query: What causes downy mildew in basil and what are the symptoms?
Found 2 relevant documents using the Search Engine.

Generated Answer:
Downy mildew in basil is caused by *Peronospora belbahrii*. The provided context does not describe the specific symptoms of downy mildew.

✅ Successfully saved the query and answer to Firebase.

--- Fetching Search History from Firebase ---
Found 16 saved searches in the database!

Cloud ID: -OtnPjTBxp8IbGIcQpgs
Query: What causes downy mildew in basil and what are the symptoms?
----------------------------------------
Cloud ID: -OtnPjU6U3YvVEXWCCCG
Query: How does temperature affect greenhouse diseases?
----------------------------------------
Cloud ID: -Otseq2--Et4DsdxiNiS
Query: What causes fusarium wilt in basil?
----------------------------------------
Cloud ID: -OtserDlJxmGEqYU3g2f
Query: What affects white mould infection in greenhouse basil?
----------------------------------------
Cloud ID: 

## 5. Cleaning Duplicate RAG Entries from Firebase



In [9]:
print("--- Fetching all RAG search results from Firebase ---")

if 'FBconn' not in globals():
    print("Error: 'FBconn' is not defined. Please run the setup cell first.")
else:
    # Fetch all results
    all_rag_results = FBconn.get('/rag_search_results', None)

    if not all_rag_results:
        print("No RAG search results found in Firebase.")
    else:
        print(f"Found {len(all_rag_results)} total entries.")

        unique_queries = {}
        entries_to_delete = []

        # Find duplicates by comparing the query strings
        for entry_id, entry_data in all_rag_results.items():
            query = entry_data.get('query')
            if query:
                normalized_query = query.lower().strip()
                if normalized_query not in unique_queries:
                    unique_queries[normalized_query] = entry_id
                else:
                    entries_to_delete.append(entry_id)

        # Delete duplicates using the custom wrapper
        if not entries_to_delete:
            print("No duplicate entries found. Database is clean. ✨")
        else:
            print(f"Found {len(entries_to_delete)} duplicate entries to delete.")
            print("Deleting duplicate entries...")
            for duplicate_id in entries_to_delete:
                try:
                    # Correct syntax for our custom wrapper
                    FBconn.delete('/rag_search_results', duplicate_id)
                    print(f"  Deleted duplicate entry with ID: {duplicate_id}")
                except Exception as e:
                    print(f"  Error deleting {duplicate_id}: {e}")

            print("Duplicate cleaning complete. Verifying...")

        # Display the final, clean unique entries
        print("\n--- Remaining Unique RAG Entries ---")
        remaining_results = FBconn.get('/rag_search_results', None)

        if remaining_results:
            print(f"Found {len(remaining_results)} unique entries in the cloud:")
            for entry_id, entry_data in remaining_results.items():
                query = entry_data.get('query', 'N/A')
                answer = entry_data.get('answer', 'N/A')
                print(f"\n🔑 Entry ID: {entry_id}")
                print(f"❓ Query: {query}")
                # We truncate the answer to keep the output readable
                print(f"💡 Answer: {answer[:100]}...")
        else:
            print("No entries remain in Firebase after cleaning.")

print("\n✅ Database Maintenance Finished.")

--- Fetching all RAG search results from Firebase ---
Found 16 total entries.
Found 1 duplicate entries to delete.
Deleting duplicate entries...
  Deleted duplicate entry with ID: -OvoAlWOPeaeisSRZ7TG
Duplicate cleaning complete. Verifying...

--- Remaining Unique RAG Entries ---
Found 15 unique entries in the cloud:

🔑 Entry ID: -OtnPjTBxp8IbGIcQpgs
❓ Query: What causes downy mildew in basil and what are the symptoms?
💡 Answer: Based on the provided context, downy mildew in basil is caused by the oomycete *Peronospora belbahri...

🔑 Entry ID: -OtnPjU6U3YvVEXWCCCG
❓ Query: How does temperature affect greenhouse diseases?
💡 Answer: Based on the provided context, temperature significantly affects greenhouse diseases, specifically i...

🔑 Entry ID: -Otseq2--Et4DsdxiNiS
❓ Query: What causes fusarium wilt in basil?
💡 Answer: Based on the provided context, the specific biological pathogen that causes Fusarium wilt is not exp...

🔑 Entry ID: -OtserDlJxmGEqYU3g2f
❓ Query: What affects white mo

## Global Automation Engine

In [10]:
# Create a dedicated display area for the video and alerts
irrigation_video_out = widgets.Output()
display(irrigation_video_out)

def trigger_irrigation_event(reason):
    """
    This function is triggered only when there is a real demand for irrigation
    (e.g., daily task, sensor reading, or AI recommendation).
    It pops up a visual alert and plays the physical irrigation video.
    """
    with irrigation_video_out:
        clear_output()

        # 1. Display Alert Popup
        alert_html = f"""
        <div style='background-color: #e1f5fe; border-left: 6px solid #0288d1; padding: 15px; margin-bottom: 15px; border-radius: 5px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);'>
            <h3 style='color: #0288d1; margin: 0; font-family: sans-serif;'>💧 Automated Irrigation System Activated!</h3>
            <p style='margin: 5px 0 0 0; color: #01579b; font-family: sans-serif; font-size: 16px;'><b>Trigger Reason:</b> {reason}</p>
        </div>
        """
        display(HTML(alert_html))

        # 2. Load the physical irrigation video (with autoplay=1)
        vimeo_video_id = "1200090024"
        video_html = f"""
        <iframe src="https://player.vimeo.com/video/{vimeo_video_id}?autoplay=1"
        width="640" height="360"
        frameborder="0"
        allow="autoplay; fullscreen; picture-in-picture"
        allowfullscreen>
        </iframe>
        """
        display(HTML(video_html))

# --- Examples of how to trigger this from within the system ---
# To test this in the notebook, here is a mock button.
# In practice, you can call the `trigger_irrigation_event` function from anywhere in your code!

test_btn = widgets.Button(
    description='🧪 Test: Trigger Irrigation',
    button_style='info',
    icon='tint'
)

def on_test_click(b):
    trigger_irrigation_event("Real sensor detection - soil moisture dropped below 40%")

test_btn.on_click(on_test_click)
display(test_btn)

Output()

Button(button_style='info', description='🧪 Test: Trigger Irrigation', icon='tint', style=ButtonStyle())

## Screens

### Screen 0: Welcome / Onboarding

In [11]:
onboarding_html = widgets.HTML(value=f"""
<div style='background:{COLOR_SURFACE}; border-left:6px solid {COLOR_PRIMARY};
            border-radius:10px; padding:24px; margin-bottom:20px; font-family:sans-serif;'>
    <h2 style='color:{COLOR_SECONDARY}; margin-top:0;'>🌱 Welcome to TurtleGrow</h2>
    <p style='color:{COLOR_TEXT}; font-size:15px; line-height:1.7;'>
        TurtleGrow is a smart cloud-based plant monitoring dashboard.<br>
        Here is what you can do:
    </p>
    <ol style='color:{COLOR_TEXT}; font-size:15px; line-height:2;'>
        <li>📸 <b> Smart Diagnosis:</b> Upload a plant photo → get an AI health report.</li>
        <li>📡 <b> IoT Sensor Data:</b> Fetch live temperature, humidity & soil readings.</li>
        <li>🧠 <b> Agronomist AI:</b> Ask any plant-disease question and get expert answers.</li>
        <li>📊 <b> Analytics Dashboard:</b> View trends and sensor history at a glance.</li>
        <li>🎮 <b> Gamification Hub:</b> Complete daily tasks, earn points, climb the leaderboard.</li>
    </ol>
</div>
""")
display(onboarding_html)


HTML(value="\n<div style='background:#f8f9fa; border-left:6px solid #2ecc71;\n            border-radius:10px; …

### Screen 1: Image Upload & AI Analysis


In [12]:
instructions = widgets.HTML(
    value=f"""
    <div style='border-left:4px solid {COLOR_PRIMARY}; padding:10px 16px;
                background:{COLOR_SURFACE}; border-radius:6px; font-family:sans-serif;'>
        <h4 style='color:{COLOR_SECONDARY}; margin:0 0 14px;'>👨‍🌾 Smart Diagnosis Tool</h4>
        <p style='color:{COLOR_TEXT}; margin:0; font-size:16px;'>
            Upload a picture of your plant. The AI will:<br>
            1. Identify visual disease symptoms (Hugging Face Vision model)<br>
            2. Fetch live sensor data (temperature, humidity, soil)<br>
            3. Generate a full agronomist report (Gemini AI)
        </p>
    </div>"""
)

upload_btn = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='📸 Upload Image',
    button_style='info',
    layout=widgets.Layout(width='300px')
)

status_label = widgets.HTML(value="<span style='color:gray;'><b>Status:</b> Waiting for image upload... 😴</span>")
out_analysis = widgets.Output()

def get_latest_sensor_data():
    """Fetches real-time IoT data from the Render server. Logs errors silently."""
    temp, hum, soil = "Unavailable", "Unavailable", "Unavailable"
    try:
        t_resp = requests.get(f"{BASE_URL}/history", params={"feed": "temperature", "limit": 1}, timeout=60).json()
        h_resp = requests.get(f"{BASE_URL}/history", params={"feed": "humidity", "limit": 1}, timeout=60).json()
        s_resp = requests.get(f"{BASE_URL}/history", params={"feed": "soil", "limit": 1}, timeout=60).json()

        if "data" in t_resp and len(t_resp["data"]) > 0:
            temp = float(t_resp["data"][0]["value"])
        if "data" in h_resp and len(h_resp["data"]) > 0:
            hum = float(h_resp["data"][0]["value"])
        if "data" in s_resp and len(s_resp["data"]) > 0:
            soil = float(s_resp["data"][0]["value"])

        logger.info(f"IoT Data fetched successfully: Temp={temp}, Hum={hum}, Soil={soil}")

    except Exception as e:
        logger.error(f"IoT Sensor Warning: Could not fetch data. Details: {e}")

    return temp, hum, soil

def analyze_image_locally(image_bytes):
    """Analyzes the image using the locally loaded Hugging Face model instead of the API."""
    try:
        # 1. Convert the raw bytes back into an Image object that the model can "see"
        img = Image.open(io.BytesIO(image_bytes)).convert("RGB")

        # 2. Run the image through our local model
        predictions = disease_classifier(img)

        # 3. Extract the highest probability result
        if predictions and len(predictions) > 0:
            best_prediction = predictions[0]['label']
            logger.info(f"Local Vision successful. Prediction: {best_prediction}")
            return best_prediction

        return "Unknown condition"

    except Exception as e:
        logger.error(f"Local Vision Error: {e}")
        return f"Analysis Unavailable (Local Error: {str(e)})"

def on_image_upload(change):
    if not upload_btn.value:
        return

    with out_analysis:
        clear_output()
        status_label.value = "<span style='color:blue;'><b>Status:</b> ⏳ Loading image...</span>"
        logger.info("New image upload triggered.")

        try:
            # Handle different colab file upload dictionary formats
            uploaded_file = upload_btn.value
            if isinstance(uploaded_file, dict):
                filename = list(uploaded_file.keys())[0]
                content = uploaded_file[filename]['content']
            else:
                content = uploaded_file[0]['content'] if isinstance(uploaded_file[0], dict) else uploaded_file[0].content

            # Display the uploaded image
            img = Image.open(io.BytesIO(content)).convert("RGB")
            img.thumbnail((300, 300))
            display(img)

            # 1. Computer Vision Analysis (Hugging Face)
            status_label.value = "<span style='color:blue;'><b>Status:</b> 🔍 Identifying visual symptoms via Cloud AI... (May take 45s)</span>"
            hf_disease = analyze_image_locally(content)

            # 2. IoT Sensor Data Retrieval
            status_label.value = "<span style='color:blue;'><b>Status:</b> 📡 Fetching live environmental data... (May take 60s if server sleeps)</span>"
            current_temp, current_hum, current_soil = get_latest_sensor_data()

            # 3. Formulating RAG Prompt
            status_label.value = "<span style='color:blue;'><b>Status:</b> 🧠 Formulating Agronomist Report...</span>"

            rag_prompt = f"""
            You are an expert agronomist AI for the TurtleGrow system.
            The Vision model identified visual symptoms as: '{hf_disease}'.
            Live IoT sensors report: Temperature {current_temp}°C, Humidity {current_hum}%, Soil Moisture {current_soil}%.

            Analyze the plant's condition based on the visual symptoms and the environmental data.
            If data is marked as 'Unavailable' or shows an 'Error', state that you cannot provide insights for that specific parameter.
            Provide a short, professional diagnostic report strictly addressing these 4 points:
            1. Plant Disease Identification.
            2. Water Stress Assessment.
            3. Pest Identification (if any visual or environmental clues suggest it).
            4. Irrigation and Treatment Recommendations.
            """

            # Ask Gemini (RAG Engine)
            logger.info("Sending prompt to Gemini LLM.")
            response = rag_instance.gemini_model.generate_content(rag_prompt)

            # Update UI to success
            status_label.value = "<span style='color:green;'><b>Status:</b> ✅ Analysis Complete!</span>"
            logger.info("Pipeline completed successfully.")

            # 4. Improved UI Output (HTML Dashboard Style)
            formatted_response = re.sub(r'\*\*(.*?)\*\*', r'<b>\1</b>', response.text)
            formatted_response = formatted_response.replace('\n', '<br>')

            report_html = f"""
            <div style="background-color: #f8f9fa; padding: 20px; border-radius: 10px; border-left: 6px solid #2ecc71; box-shadow: 0 4px 8px rgba(0,0,0,0.1); margin-top: 15px;">
                <h2 style="color: #27ae60; margin-top: 0; font-family: sans-serif;">🌿 AI Smart Diagnosis Report</h2>
                <p style="font-family: sans-serif; font-size: 16px;"><b>👁️ Visual Identification:</b> {hf_disease}</p>
                <p style="font-family: sans-serif; font-size: 16px;"><b>🌡️ Current Environment:</b> Temp {current_temp}°C | Humidity {current_hum}% | Soil {current_soil}%</p>
                <hr style="border: 0; height: 1px; background-color: #dcdde1; margin: 15px 0;">
                <h4 style="color: #2c3e50; font-family: sans-serif; margin-bottom: 10px;">Expert Agronomist Assessment:</h4>
                <div style="font-family: sans-serif; font-size: 15px; line-height: 1.6; color: #34495e;">
                    {formatted_response}
                </div>
            </div>
            """
            display(widgets.HTML(report_html))

        except Exception as e:
            status_label.value = f"<span style='color:red;'><b>Status:</b> ❌ Error: {str(e)}</span>"
            logger.critical(f"Critical Error in execution pipeline: {e}", exc_info=True)

# Attach event listener to the upload button
upload_btn.observe(on_image_upload, names='value')

# Display Screen layout
display(widgets.VBox([
    instructions,
    upload_btn,
    status_label,
    out_analysis
]))

### Screen 2: IoT Sensor Data


In [13]:
instructions_iot = widgets.HTML(
    value="<h4>📡 Live Field Data</h4><p>Fetch real-time metrics from the remote agricultural sensors.</p>"
)

# User selects which feed to read
feed_dropdown = widgets.Dropdown(
    options=[
        ("💧 Humidity", "humidity"),
        ("🌱 Soil Moisture", "soil"),
        ("🌡️ Temperature", "temperature"),
        ("📦 All Data (Full Record)", "json")
    ],
    value="temperature",
    description="📡 Sensor:",
    layout=widgets.Layout(width='300px')
)

# User selects how many samples to fetch backward
limit_slider = widgets.IntSlider(
    value=10, min=1, max=50, step=1,
    description="📊 Samples:",
    layout=widgets.Layout(width='300px')
)

fetch_btn = widgets.Button(
    description="📥 Fetch Data & Save",
    button_style="primary",
    layout=widgets.Layout(width='300px'),
    icon='cloud-download'
)

samples_tooltip = widgets.HTML(value="<span style='color:gray; font-size:12px;'>ℹ️ Number of recent sensor readings to display (e.g. 10 = last 10 readings)</span>")
# Status label for informative feedback- No code errors shown to user
status_label_iot = widgets.HTML(value="<span style='color:gray;'><b>Status:</b> Ready to fetch data.</span>")
out_iot = widgets.Output()

def fetch_iot_data(b):
    with out_iot:
        clear_output()
        feed = feed_dropdown.value
        limit = limit_slider.value

        # Get the friendly display name from the dropdown options based on the selected value
        display_name = next(opt[0] for opt in feed_dropdown.options if opt[1] == feed)

        # Clean the display name by removing the emoji for the table header
        clean_display_name = display_name.split(" ", 1)[1] if " " in display_name else display_name

        # Friendly UI message mentioning the potential Render server delay
        status_label_iot.value = f"<span style='color:blue;'><b>Status:</b> ⏳ Waking up IoT server and fetching '{clean_display_name}' data... (May take up to 60s)</span>"
        logger.info(f"Screen 2: User requested {limit} samples for feed: {feed}")

        try:
            # We explicitly use timeout=60 to handle the Render server "Cold Start" sleep time
            response = requests.get(f"{BASE_URL}/history", params={"feed": feed, "limit": limit}, timeout=60)

            if response.status_code == 200:
                data = response.json()

                if "data" in data and len(data["data"]) > 0:
                    status_label_iot.value = f"<span style='color:green;'><b>Status:</b> ✅ Successfully retrieved {len(data['data'])} samples!</span>"
                    logger.info(f"Screen 2: Successfully fetched {len(data['data'])} records.")

                    # Convert to DataFrame for a pretty table display
                    df = pd.DataFrame(data["data"])

                    if feed == "json":
                        # Parsing JSON and adding units to keys ---
                        def format_json_with_units(val):
                            # Ensure it's a dict if it comes as a string
                            import json
                            d = json.loads(val) if isinstance(val, str) else val

                            formatted_d = {}
                            for k, v in d.items():
                                if k == 'temperature': formatted_d[k] = f"{float(v):.2f}°C"
                                elif k == 'humidity': formatted_d[k] = f"{float(v):.2f}%"
                                elif k == 'soil': formatted_d[k] = f"{float(v):.2f}%"
                                else: formatted_d[k] = v
                            return str(formatted_d).replace("'", '"') # Make it look like standard JSON

                        df["value"] = df["value"].apply(format_json_with_units)

                    else:
                        # Regular sensors
                        units = {"humidity": "%", "soil": "%", "temperature": "°C"}
                        unit = units.get(feed, "")
                        df["value"] = pd.to_numeric(df["value"], errors="coerce").round(1)
                        df["value"] = df["value"].astype(str) + " " + unit

                    # Format the datetime for better readability
                    df['created_at'] = pd.to_datetime(df['created_at']).dt.strftime('%Y-%m-%d %H:%M:%S')

                    # Start index from 1 instead of 0
                    df.index = df.index + 1

                    # Display the data nicely using the clean display name
                    display(widgets.HTML(f"<b>Recent {clean_display_name} Readings:</b>"))

                    # Prepare the table for display
                    df_display = df[['created_at', 'value']].head(limit)

                    # Expand and wrap text if it's a long JSON string ---
                    if feed == "json":
                        # JSON data: Left-aligned text (for readability), Centered headers
                        styled_df = df_display.style.set_properties(**{
                            'text-align': 'left',
                            'white-space': 'pre-wrap',
                            'min-width': '400px'
                        }).set_table_styles([
                            {'selector': 'th', 'props': [('text-align', 'center')]}
                        ])
                        display(styled_df)
                    else:
                        # Regular sensors: Centered data and Centered headers
                        styled_df = df_display.style.set_properties(**{
                            'text-align': 'center'
                        }).set_table_styles([
                            {'selector': 'th', 'props': [('text-align', 'center')]}
                        ])
                        display(styled_df)

                    # --- Automation Feature Integration ---
                    # Trigger the irrigation event if the soil moisture drops below 40%
                    if feed == "soil":
                        try:
                            # Extract the numeric value from the formatted string (e.g., "38.5 %" -> 38.5)
                            latest_soil_val = float(str(df["value"].iloc[0]).split()[0])

                            if latest_soil_val < 40.0:
                                trigger_irrigation_event(f"Sensor detection: Soil moisture critically low at {latest_soil_val}%")

                        except NameError:
                            print("⚠️ Note: Please run the Global Automation Engine cell first to enable automation.")
                        except Exception as e:
                            logger.error(f"Automation trigger error: {e}")
                    # ------------------------------------------

                    # Save the latest reading to Firebase DB for our internal system
                    try:
                        latest_val = str(df["value"].iloc[0]) if feed == "json" else df["value"].iloc[0]

                        sensor_record = {
                            "sensor_type": feed,
                            "latest_value": latest_val,
                            "timestamp": datetime.datetime.now().isoformat()
                        }
                        FBconn.post('/iot_sensor_data/', sensor_record)
                        logger.info("Screen 2: Latest sensor value backed up to Firebase.")

                    except Exception as fb_err:
                        # Log Firebase errors silently, don't ruin the user experience
                        logger.error(f"Screen 2: Failed to backup to Firebase: {fb_err}")

                else:
                    # Server responded, but Adafruit had no data for this feed
                    status_label_iot.value = "<span style='color:orange;'><b>Status:</b> ⚠️ Server is awake, but no data was found for this specific sensor.</span>"
                    logger.warning("Screen 2: IoT Server returned an empty data array.")
            else:
                # HTTP errors (e.g., 500 Internal Server Error)
                status_label_iot.value = "<span style='color:red;'><b>Status:</b> ❌ Remote server error. Please try again later.</span>"
                logger.error(f"Screen 2: IoT Server returned HTTP status code {response.status_code}")

        except requests.exceptions.Timeout:
            # Specific handling for if the 60 seconds passed and the server didn't wake up
            status_label_iot.value = "<span style='color:red;'><b>Status:</b> ❌ Server is taking too long to wake up. Please try again.</span>"
            logger.error("Screen 2: Request to Render server timed out after 60 seconds.")

        except Exception as e:
            # Catch-all for network issues, DNS errors, etc.
            status_label_iot.value = "<span style='color:red;'><b>Status:</b> ❌ Network connection error. Please check your internet.</span>"
            logger.error(f"Screen 2: Critical Connection Error: {e}", exc_info=True)

# Connect button to function
fetch_btn.on_click(fetch_iot_data)

# Display the complete UI block
display(widgets.VBox([
    instructions_iot,
    feed_dropdown,
    limit_slider,
    samples_tooltip,
    fetch_btn,
    status_label_iot,
    out_iot
]))

### Screen 3: RAG Search Engine Query


In [14]:
# UI Elements Setup for RAG Search
query_input = widgets.Text(
    value='',
    placeholder='e.g., What causes black spot in sweet basil?',
    description='❓ Ask:',
    layout=widgets.Layout(width='80%')
)

search_btn = widgets.Button(
    description='🧠 Ask the Expert (RAG)',
    button_style='warning',
    icon='search'
)

out_search = widgets.Output()

def perform_search(b):
    with out_search:
        clear_output()
        q = query_input.value
        if not q:
            print("⚠️ Please type a question in the text box.")
            return

        print(f"🔎 Analyzing query: '{q}'...\n")

        # 1. Search the Inverted Index for relevant keywords
        keywords = re.findall(r'\b[a-zA-Z]+\b', q)
        retrieved_context = rag_instance.retrieve_documents(keywords, top_n=2)

        # 2. Generate the smart answer using the Gemini LLM
        answer = rag_instance.generate_rag_response(q, retrieved_context)

        # 3. Display the final generated response
        print("💡 Expert Answer:")
        print("-" * 50)
        display(Markdown(answer))
        print("-" * 50)
        print("\n📚 Sources:")

        # Dictionary mapping document IDs to their actual academic titles
        article_titles = {
            1: "Characterization of Colletotrichum ocimi (Black Spot) in Sweet Basil",
            2: "Epidemics of Fusarium wilt and black spot on basil",
            3: "Susceptibility of Sweet Basil to Downy Mildew",
            4: "Staining method for rapid detection of basil downy mildew",
            5: "PCR analysis of P. belbahrii on sweet basil seed"
        }

        # Display dynamically generated clickable hyperlinks for the sources
        for doc in retrieved_context:
            doc_id = doc['id']
            # Retrieve the original URL from the RAG system's mapping
            url = rag_instance.doc_id_mapping.get(doc_id, "#")
            # Retrieve the human-readable title or fallback to a generic name
            title = article_titles.get(doc_id, f"Article {doc_id}")

            # Render the Markdown link format: [Title](URL)
            display(Markdown(f"- [{title}]({url})"))

        # 4. Save the query and answer to Firebase for historical tracking and caching
        try:
            context_text = "\n".join([f"Doc {doc['id']}: {doc['text']}" for doc in retrieved_context])
            rag_data_new = {
                'query': q,
                'answer': answer,
                'context': context_text,
                'timestamp': datetime.datetime.now().isoformat()
            }
            FBconn.post('/rag_search_results/', rag_data_new)
        except Exception as e:
            # Fail silently to maintain a clean UI for the user
            pass

# Attach the click event listener to the search button
search_btn.on_click(perform_search)

# Display the complete UI layout for the Search Screen
display(widgets.VBox([
    widgets.HTML("<h3>🧠 TurtleGrow: Agronomist AI Assistant</h3>"),
    widgets.HBox([query_input, search_btn]),
    out_search
]))

### Screen 4: Visual Dashboard


In [15]:
status_label_dash = widgets.HTML(value="<span style='color:gray;'>Initializing dashboard...</span>")
out_dash = widgets.Output()

def draw_dashboard(b=None):
    with out_dash:
        clear_output()
        # Update user status UI
        status_label_dash.value = "<span style='color:blue;'>⏳ Loading metrics...</span>"
        logger.info("Dashboard: Auto-loading data started.")

        curr_temp, curr_hum, curr_soil = "N/A", "N/A", "N/A"

        # 1. Fetch data
        try:
            # Concurrent-like fetching (sequential requests to the central server)
            t_req = requests.get(f"{BASE_URL}/history", params={"feed": "temperature", "limit": 1}, timeout=60).json()
            h_req = requests.get(f"{BASE_URL}/history", params={"feed": "humidity", "limit": 1}, timeout=60).json()
            s_req = requests.get(f"{BASE_URL}/history", params={"feed": "soil", "limit": 1}, timeout=60).json()

            if "data" in t_req and len(t_req["data"]) > 0: curr_temp = f'{t_req["data"][0]["value"]}°C'
            if "data" in h_req and len(h_req["data"]) > 0: curr_hum = f'{h_req["data"][0]["value"]}%'
            if "data" in s_req and len(s_req["data"]) > 0: curr_soil = f'{s_req["data"][0]["value"]}%'

            logger.info("Dashboard: Successfully fetched latest stats.")
            status_label_dash.value = "<span style='color:green;'>✅ Data loaded successfully.</span>"

        except Exception as e:
            logger.error(f"Dashboard: Failed to fetch latest stats: {e}", exc_info=True)
            status_label_dash.value = "<span style='color:red;'>⚠️ Connection issue. Check logs.</span>"

        # 2. Design "Quick Stats" cards  (Fix: unified colors from COLOR_* palette)
        stats_html = f"""
        <div style="display: flex; gap: 20px; margin-bottom: 20px;">
            <div style="background:{COLOR_SURFACE}; padding:15px; border-radius:10px; text-align:center; flex:1; border-top:4px solid {COLOR_DANGER};">
                <h4 style="margin:0; color:{COLOR_TEXT};">🌡️ Temperature</h4>
                <h2 style="margin:10px 0; color:{COLOR_DANGER};">{curr_temp}</h2>
            </div>
            <div style="background:{COLOR_SURFACE}; padding:15px; border-radius:10px; text-align:center; flex:1; border-top:4px solid {COLOR_ACCENT};">
                <h4 style="margin:0; color:{COLOR_TEXT};">💧 Humidity</h4>
                <h2 style="margin:10px 0; color:{COLOR_ACCENT};">{curr_hum}</h2>
            </div>
            <div style="background:{COLOR_SURFACE}; padding:15px; border-radius:10px; text-align:center; flex:1; border-top:4px solid {COLOR_WARNING};">
                <h4 style="margin:0; color:{COLOR_TEXT};">🌱 Soil</h4>
                <h2 style="margin:10px 0; color:{COLOR_WARNING};">{curr_soil}</h2>
            </div>
        </div>
        """
        display(widgets.HTML(stats_html))

        # 3. Draw graph
        try:
            resp = requests.get(f"{BASE_URL}/history", params={"feed": "temperature", "limit": 20}, timeout=60)
            data = resp.json()

            if "data" in data and len(data["data"]) > 0:
                df = pd.DataFrame(data["data"])
                df["created_at"] = pd.to_datetime(df["created_at"])
                df["value"] = pd.to_numeric(df["value"], errors="coerce")
                df = df.iloc[::-1].reset_index(drop=True)

                fig, ax = plt.subplots(figsize=(10, 4))
                ax.plot(df["created_at"], df["value"], marker='o', color='#f39c12', linewidth=2)
                ax.fill_between(df["created_at"], df["value"], alpha=0.1, color='#f39c12')
                ax.set_title('Leaf Temperature History', fontsize=12)
                ax.grid(True, linestyle='--', alpha=0.5)
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)
                plt.xticks(rotation=45)
                plt.tight_layout()
                plt.show()
                logger.info("Dashboard: Graph rendered successfully.")
            else:
                logger.warning("Dashboard: No trend data found to plot.")
        except Exception as e:
            logger.error(f"Dashboard: Graph rendering error: {e}")

# Layout
display(widgets.VBox([
    widgets.HTML("<h2>🌿 Analytics Dashboard</h2>"),
    status_label_dash,
    out_dash
]))

# Automatically load data upon execution
draw_dashboard()

### Screen 5:Gamification Hub

In [16]:
# Fetch real data from Firebase
try:
    # Try fetching Yossi's score from the cloud
    cloud_score = FBconn.get('/users/yossi/score', None)
    if cloud_score is not None:
        current_score = int(cloud_score)
    else:
        # If no score exists, set a default initial score
        current_score = 1847
        # Save the initial score to the cloud
        FBconn.put('/users/yossi', 'score', current_score)
except Exception as e:
    # In case of an error (e.g., network issue), use a local score.
    # The error is caught silently and not displayed to the user!
    current_score = 1847

# Calculate current level: every 150 points equals one level
current_level = current_score // 150
# Current streak (in days)
streak_days = 7

def get_header_html():
    # Level is calculated dynamically based on the score
    lvl = current_score // 150
    return f"""
    <div style="display:flex; gap:15px; margin-bottom:20px;">
        <div style="background:{COLOR_WARNING}; color:white; padding:15px; border-radius:10px; flex:1; text-align:center;">
            <h4 style="margin:0;">🏆 Total Score (Level {lvl})</h4>
            <h1 style="margin:5px 0;">{current_score}</h1>
        </div>
        <div style="background:{COLOR_DANGER}; color:white; padding:15px; border-radius:10px; flex:1; text-align:center;">
            <h4 style="margin:0;">🔥 Current Streak</h4>
            <h1 style="margin:5px 0;">{streak_days} Days</h1>
        </div>
        <div style="background:{COLOR_ACCENT}; color:white; padding:15px; border-radius:10px; flex:1; text-align:center;">
            <h4 style="margin:0;">📈 Global Rank</h4>
            <h1 style="margin:5px 0;">#4</h1>
        </div>
    </div>
    """

# Header widget to display the top metrics
header_widget = widgets.HTML(value=get_header_html())
# Output area for the leaderboard
leaderboard_out = widgets.Output()

def refresh_leaderboard():
    # Function to redraw the leaderboard
    with leaderboard_out:
        clear_output()
        print("👥 LEADERBOARD (LIVE)")
        print("=" * 30)
        print("1. 🥇 GardenMaster2024  - 3,245 pts")
        print("2. 🥈 PlantWhisperer    - 2,891 pts")
        print("3. 🥉 GreenThumbPro     - 2,654 pts")
        # Yossi's score is integrated here dynamically
        print(f"4. 🔵 YOU (Yossi)       - {current_score} pts")
        print("5. 🌿 NatureNinja       - 1,723 pts")
        print("=" * 30)

# Initial call to draw the leaderboard
refresh_leaderboard()

# --- Create daily tasks and buttons ---
task_label = widgets.HTML("<h3>📋 Today's Daily Tasks:</h3>")
btn_task1 = widgets.Button(description="💧 Water morning plants (+50)", button_style="success", icon="check-circle", layout=widgets.Layout(width='300px'))
btn_task2 = widgets.Button(description="🔍 Check soil moisture (+30)", button_style="info", icon="check-circle", layout=widgets.Layout(width='300px'))
btn_task3 = widgets.Button(description="✂️ Prune herbs (+45)", button_style="warning", icon="check-circle", layout=widgets.Layout(width='300px'))

# System message for the user
sys_message = widgets.HTML("<p style='color:gray;'><i>Complete tasks to earn points and climb the leaderboard!</i></p>")

def complete_task(b, points):
    # Function triggered when a task is clicked
    global current_score
    # Add points
    current_score += points

    # ---  Automation Feature ---
    # If the completed task is worth 50 points, it's the watering task,
    # so we trigger the irrigation video and alert.
    if points == 50:
        try:
            trigger_irrigation_event("Completed Daily Task: Water morning plants")
        except NameError:
            print("⚠️ Note: Please run the cell containing trigger_irrigation_event() first.")
    # -----------------------------------

    # --- Real cloud saving happens silently in the background ---
    try:
        # Update the new score in Firebase
        FBconn.put('/users/yossi', 'score', current_score)
    except Exception as e:
        pass # We do not bother the user with server errors
    # -------------------------------------

    # Disable the button after clicking to prevent double clicks
    b.disabled = True
    b.description = f"Done! (+{points})"
    b.button_style = "" # Remove button color to indicate completion

    # Update on-screen widgets
    header_widget.value = get_header_html()
    refresh_leaderboard()
    # Clean and encouraging message for the user
    sys_message.value = f"<p style='color:green; font-weight:bold;'>🎉 Task completed! You earned {points} points.</p>"

# Link buttons to the function with the appropriate points for each task
btn_task1.on_click(lambda b: complete_task(b, 50))
btn_task2.on_click(lambda b: complete_task(b, 30))
btn_task3.on_click(lambda b: complete_task(b, 45))

# --- Screen layout: left column (tasks) and right column (leaderboard) ---
left_column = widgets.VBox([task_label, btn_task1, btn_task2, btn_task3, sys_message])
right_column = widgets.VBox([leaderboard_out])
main_layout = widgets.HBox([left_column, right_column], layout=widgets.Layout(justify_content='space-around'))

# Display the entire screen layout
display(widgets.VBox([
    widgets.HTML("<h2>🎮 Gamification Hub</h2>"),
    header_widget,
    main_layout
]))

### Application

In [17]:
# ==============================================================================
# 📱 MAIN APPLICATION LAYOUT (SINGLE CELL APP)
# ==============================================================================

# 1. Package each screen into an independent container (VBox) instead of rendering them directly
screen_0_layout = widgets.VBox([onboarding_html])

screen_1_layout = widgets.VBox([instructions, upload_btn, status_label, out_analysis])

screen_2_layout = widgets.VBox([instructions_iot, feed_dropdown, limit_slider, samples_tooltip, fetch_btn, status_label_iot, out_iot])

screen_3_layout = widgets.VBox([widgets.HTML("<h3>🧠 TurtleGrow: Agronomist AI Assistant</h3>"), widgets.HBox([query_input, search_btn]), out_search])

screen_4_layout = widgets.VBox([widgets.HTML("<h2>🌿 Analytics Dashboard</h2>"), status_label_dash, out_dash])

screen_5_layout = widgets.VBox([widgets.HTML("<h2>🎮 Gamification Hub</h2>"), header_widget, main_layout])

# 2. Create the main tabbed navigation container and inject the screen layouts
app_tabs = widgets.Tab(children=[
    screen_0_layout,
    screen_1_layout,
    screen_2_layout,
    screen_3_layout,
    screen_4_layout,
    screen_5_layout
])

# 3. Configure the visible titles for each corresponding tab item
app_tabs.set_title(0, '🏠 Home')
app_tabs.set_title(1, '📸 AI Diagnosis')
app_tabs.set_title(2, '📡 IoT Sensors')
app_tabs.set_title(3, '🧠 Expert Q&A')
app_tabs.set_title(4, '📊 Dashboard')
app_tabs.set_title(5, '🎮 Gamification')

# 4. Trigger the automatic initial data fetch for the Analytics Dashboard (Screen 4) on startup
draw_dashboard()

# 5. Execute the single unified display call to render the entire modular application
display(app_tabs)